# A5 · Statistical Tests

*Path A: Write It Yourself, notebook 5 of 6 · about 20 to 30 minutes*

**In this notebook you will learn to:**

- Choose between a t-test, ANOVA, and a chi-square test
- Run each one in R
- Find the important numbers in the output

**Before this notebook:** [A4: Exploring and Plotting](https://colab.research.google.com/github/icohr-lab/tutorials/blob/main/notebooks/A4_Exploring_and_Plotting.ipynb). This notebook assumes you have met these tests in a statistics course; it shows how to run them, not the theory behind them.

> **Before you start:** click **File > Save a copy in Drive** and work in your copy. Run a code cell by clicking the play button to its left, or press **Shift + Enter**. Go in order from top to bottom.

## 1. Setup

Run the setup cell. The lines printed by `library(tidyverse)` and `read_csv()` are normal.

In [ ]:
library(tidyverse)

study_df <- read_csv("https://raw.githubusercontent.com/icohr-lab/aou-workshop/refs/heads/main/data/final_dataset2.csv")

## 2. Which test?

The choice depends on what kind of variables you are comparing.

| Outcome | Compared across | Test | R function |
|---|---|---|---|
| A number (e.g. blood pressure) | 2 groups | Two-sample t-test | `t.test()` |
| A number | 3 or more groups | One-way ANOVA | `aov()` |
| A category (e.g. disease yes/no) | Groups | Chi-square test | `chisq.test()` |

## 3. Two-sample t-test

**Question:** is mean systolic blood pressure different in people with and without the condition?

Always look at the numbers before testing.

In [ ]:
study_df %>%
  group_by(condition_status) %>%
  summarise(n = n(), mean_SBP = mean(SBP, na.rm = TRUE))

The two means are almost the same. (`n` counts everyone in each group, including the few people with missing blood pressure, who are left out of the mean and of the test.) Now the test. Inside `t.test()` is a **formula**: the outcome, a tilde `~`, and the grouping variable. Read `SBP ~ condition_status` as "SBP by condition status". `data =` says which dataframe the columns come from.

In [ ]:
t.test(SBP ~ condition_status, data = study_df)

How to read the output:

- **`p-value`**: 0.84. There is no evidence of a difference in mean blood pressure between the groups.
- **`95 percent confidence interval`**: the range of values for the true difference between the two means (group 0 minus group 1) that are compatible with the data. It runs from about -5 to 6, and includes 0.
- **`sample estimates`**: the two group means.
- **`Welch Two Sample t-test`**: by default R uses the Welch version, which does not assume the two groups have equal variance. It is generally the safer choice for clinical data.

**Your turn 1.** Is mean `age` different in people with and without the condition? Look at the group means, then run the t-test.

*Expected result: means 51.4 and 56.4; p-value 0.24*

In [ ]:
# Your code here

## 4. One-way ANOVA

**Question:** is mean blood pressure different across the three `sex_at_birth` groups? With more than two groups, use ANOVA.

It takes two steps: `aov()` fits the model, and `summary()` shows the table.

In [ ]:
anova_model <- aov(SBP ~ sex_at_birth, data = study_df)
summary(anova_model)

The p-value is in the column `Pr(>F)`: 0.661. There is no evidence that mean blood pressure differs between the groups. The last line notes that 5 people were left out because their blood pressure is missing.

**Your turn 2.** Is mean `age` different across the `race` groups? Run a one-way ANOVA.

*Expected result: p-value 0.814*

In [ ]:
# Your code here

## 5. Chi-square test

**Question:** is drug exposure associated with the condition? Both variables are categories (yes/no), so use a chi-square test.

Start with the table of counts. `table()` with two columns makes a cross-tabulation: the first column gives the rows, the second gives the columns.

In [ ]:
exposure_table <- table(study_df$exposure_status, study_df$condition_status)
exposure_table

Rows are exposure (0 = unexposed, 1 = exposed); columns are the condition (0 = no, 1 = yes). Of the 65 unexposed, 9 have the condition. Of the 35 exposed, 21 do. `chisq.test()` takes this table.

In [ ]:
chisq.test(exposure_table)

The p-value is written `4.76e-06`. This is scientific notation: `e-06` means "move the decimal point 6 places to the left", so the value is 0.00000476. That is very small: strong evidence that exposure and the condition are associated in this (simulated) data.

The heading of the output mentions "Yates' continuity correction". For a table with two rows and two columns, R applies this small adjustment by default, so the p-value is slightly different from one calculated by hand without it.

### When counts are small

The chi-square test is unreliable when the **expected** count in some cell of the table is small (a common rule of thumb is below 5). That happens when a group has very few people. R warns you when it does.

**Your turn 3.** Is `sex_at_birth` associated with the condition? Make the table and run `chisq.test()` on it. Read the warning. Then run `fisher.test()` on the same table: Fisher's exact test is the alternative to use when counts are small.

*Expected result: chi-square p-value 0.33, with a warning; Fisher's exact test p-value 0.30*

In [ ]:
# Your code here

## 6. Keeping a result

A test result can be stored in a variable, and individual numbers pulled out of it with `$`, just like columns of a dataframe.

In [ ]:
t_result <- t.test(SBP ~ condition_status, data = study_df)
t_result$p.value

> **Remember:** R will run any test you ask for, on any data, without checking that the test is appropriate. Choosing the right test, checking its assumptions, and interpreting the result are your job.

---
## Recap

| Code | What it does |
|---|---|
| `t.test(y ~ group, data = df)` | Two-sample t-test (Welch by default) |
| `summary(aov(y ~ group, data = df))` | One-way ANOVA |
| `table(df$a, df$b)` | Cross-tabulation of two columns |
| `chisq.test(tbl)` | Chi-square test on a table |
| `fisher.test(tbl)` | Fisher's exact test, for small counts |
| `result$p.value` | Pull the p-value out of a stored test result |

---
## Solutions

Try each exercise before looking. There is often more than one correct way to write it.

**Your turn 1**

```r
study_df %>%
  group_by(condition_status) %>%
  summarise(n = n(), mean_age = mean(age))

t.test(age ~ condition_status, data = study_df)
```

**Your turn 2**

```r
age_model <- aov(age ~ race, data = study_df)
summary(age_model)
```

**Your turn 3**

The warning "Chi-squared approximation may be incorrect" appears because the "Other" group has only 8 people, so the expected number of them with the condition is only 2.4. Neither test shows evidence of an association.

```r
sex_table <- table(study_df$sex_at_birth, study_df$condition_status)
sex_table
chisq.test(sex_table)
fisher.test(sex_table)
```

---
## Next

Continue with [A6: Regression](https://colab.research.google.com/github/icohr-lab/tutorials/blob/main/notebooks/A6_Regression.ipynb).